## K-Nearest Neighbors (kNN)

In [828]:
import numpy as np

$$ \text{Manhattan} = \sum_{i = 1}^{n} | x1_i - x2_i | $$

In [829]:
def manhattan(x1, x2):
    
    return np.sum(np.abs(x1 - x2))

$$ \text{Euclidiana} = \sqrt{\sum_{i = 1}^{n} (x1_i - x2_i)^2}$$

In [830]:
def euclidiana(x1, x2):
    
    return np.sqrt(np.sum((x1 - x2) ** 2))

$$ \text{Chebyshev} = \text{max}_{1 \le i \le n} | x1_i - x2_i |$$

In [831]:
def chebyshev(x1, x2):
    
    return np.max(np.abs(x1 - x2))

$$ \text{Hamming} = \sum_{j=1}^{n} \mathbb{I}(x_j \neq z_j) $$

In [832]:
def hamming (x1, x2):
    
    return np.sum(x1 != x2)

$$ \text{Normalização min-max} = \frac{x - x_{min}}{x_{max} - x_{min}} $$

In [833]:
def min_max(dados):

    dados_norm = dados.astype(float).copy()

    for i in range (dados_norm.shape[1]):
        coluna_min = min(dados_norm[:, i])
        coluna_max = max(dados_norm[:, i])

        for j in range(dados_norm.shape[0]):

            if coluna_max - coluna_min == 0:
                dados_norm[j, i] = 0.0
            else:
                dados_norm[j, i] = (dados_norm[j, i] - coluna_min) / (coluna_max - coluna_min)

    return dados_norm

In [834]:
def z_score(dados):
    dados_norm = dados.astype(float).copy()

    for i in range (dados_norm.shape[1]):
        
        media = np.mean(dados_norm[:, i])
        desvio_padrao = np.std(dados_norm[:, i])

        for j in range(dados_norm.shape[0]):
            if desvio_padrao == 0:
                dados_norm[j, i] = 0.0
            else:
                dados_norm[j, i] = (dados_norm[j, i] - media) / desvio_padrao

    return dados_norm

### Classificação com kNN

In [835]:
X_clf = np.array([
    # Classe 0 (não spam)
    [0, 0],
    [1, 0],
    [1, 1],
    [2, 0],
    [2, 1],
    [0, 1],
    [3, 0],
    [1, 2],
    [2, 2],
    [3, 1],

    # Classe 1 (spam)
    [5, 3],
    [6, 3],
    [6, 4],
    [7, 2],
    [7, 4],
    [8, 3],
    [8, 5],
    [9, 4],
    [4, 3],
    [5, 4]
])

y_clf = np.array([
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1
])

In [836]:
def classificarKNN(k, distancia, x, y, teste):
    
    distancias = np.zeros((x.shape[0], 2))

    if distancia == "euclidiana":
        for i in range (x.shape[0]):
            d = euclidiana(teste, x[i])

            # guarda a distância e a classe
            distancias[i, 0] = d
            distancias[i, 1] = y[i]
    
    if distancia == "manhattan":
        for i in range (x.shape[0]):
            d = manhattan(teste, x[i])

            # guarda a distância e a classe
            distancias[i, 0] = d
            distancias[i, 1] = y[i]

    if distancia == "chebyshev":
        for i in range (x.shape[0]):
            d = chebyshev(teste, x[i])

            # guarda a distância e a classe
            distancias[i, 0] = d
            distancias[i, 1] = y[i]

    if distancia == "hamming":
        for i in range (x.shape[0]):
            d = hamming(teste, x[i])

            # guarda a distância e a classe
            distancias[i, 0] = d
            distancias[i, 1] = y[i]

    
    # ordenando
    indices_ordenados = np.argsort(distancias[:, 0])
    distancias_ordenadas = distancias[indices_ordenados]

    # pegando os primeiros k (menores distâncias)
    k_pontos_mais_proximos = distancias_ordenadas[:k]

    # as classes e a quantidade de vezes que aparecem
    classes, contagens = np.unique(k_pontos_mais_proximos[:, 1], return_counts=True)
    
    # pega o índice que tem a maior contagem e retorna a classe
    classe_predita = classes[np.argmax(contagens)]

    return int(classe_predita)

### Testando classificação e avaliando

In [837]:
def testar_classificacao(testes, y_teste, k, distancia, x, y):

    fp = 0
    fn = 0
    vp = 0
    vn = 0
    acertos = 0


    for i in range (testes.shape[0]):
        class_prevista = classificarKNN(k, distancia, x, y, testes[i])

        if y_teste[i] == class_prevista:
            if class_prevista == 1:
                vp += 1
            else:
                vn += 1

            acertos += 1
        else:
            if class_prevista == 1:
                fp += 1
            else:
                fn += 1

    return fp, fn, vp, vn, acertos

In [838]:
# testes gerados por IA
X_teste = np.array([
    # Casos claros de não spam (0)
    [0, 0],
    [1, 0],
    [2, 0],
    [0, 1],
    [1, 1],

    # Casos claros de spam (1)
    [6, 1],
    [7, 1],
    [8, 1],
    [9, 1],
    [6, 0],

    # Casos na fronteira de decisão (bons para testar k = 1 vs. k = 3 ou k = 5)
    [3, 0],
    [2, 1],
    [3, 1],
    [4, 0],
    [4, 1],
    [5, 0]
])


# Classificação esperada (y)
y_teste = np.array([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1,
    0, 0, 1, 0, 1, 1
])

É necessário normalizar o teste também

In [839]:
def normalizar_teste_min_max(dados_teste, dados_treino):
    dados_norm = dados_teste.astype(float).copy()
    
    for i in range(dados_treino.shape[1]):
        coluna_min = np.min(dados_treino[:, i])
        coluna_max = np.max(dados_treino[:, i])
        
        faixa = coluna_max - coluna_min
        if faixa == 0:
            dados_norm[:, i] = 0.0
        else:
            dados_norm[:, i] = (dados_norm[:, i] - coluna_min) / faixa
            
    return dados_norm


def normalizar_teste_z_score(dados_teste, dados_treino):
    dados_norm = dados_teste.astype(float).copy()
    
    for i in range(dados_treino.shape[1]):
        media = np.mean(dados_treino[:, i])
        desvio_padrao = np.std(dados_treino[:, i])
        
        if desvio_padrao == 0:
            dados_norm[:, i] = 0.0
        else:
            dados_norm[:, i] = (dados_norm[:, i] - media) / desvio_padrao
            
    return dados_norm

In [840]:
x_treino_minmax = min_max(X_clf)
x_treino_zscore = z_score(X_clf)

X_teste_minmax = normalizar_teste_min_max(X_teste, X_clf)
X_teste_zscore = normalizar_teste_z_score(X_teste, X_clf)

"""
teste com diferentes normalizações
teste com diferentes distâncias: manhattan, euclidiana, chebyshev, hamming
"""

fp, fn, vp, vn, acertos = testar_classificacao(X_teste_minmax, y_teste, 3, "manhattan", x_treino_minmax, y_clf)

Matriz de confusão

In [841]:
def print_matriz_confusao(fp, fn, vp, vn):
    total = vp + vn + fp + fn
    print(f"Exemplo com {total} casos")
    print("-" * 52)
    print(f"{'':16} | {'Predição':^31}")
    print(f"{'Real':^16} | {'Positivo (1)':^14} | {'Negativo (0)':^14}")
    print("-" * 52)
    print(f"{'Positivo (1)':<16} | {f'VP = {vp}':^14} | {f'FN = {fn}':^14}")
    print(f"{'Negativo (0)':<16} | {f'FP = {fp}':^14} | {f'VN = {vn}':^14}")
    print("-" * 52)
    print("\n▶ VP: verdadeiro positivo.      ▶ FP: falso positivo.")
    print("▶ VN: verdadeiro negativo.      ▶ FN: falso negativo.")

print_matriz_confusao(fp, fn, vp, vn)

Exemplo com 16 casos
----------------------------------------------------
                 |            Predição            
      Real       |  Positivo (1)  |  Negativo (0) 
----------------------------------------------------
Positivo (1)     |     VP = 4     |     FN = 4    
Negativo (0)     |     FP = 0     |     VN = 8    
----------------------------------------------------

▶ VP: verdadeiro positivo.      ▶ FP: falso positivo.
▶ VN: verdadeiro negativo.      ▶ FN: falso negativo.


$$ \text{Acurácia(Accuracy)} = \frac{VP + VN}{VP + VN + FP + FN} $$

In [842]:
def acuracia(vp, vn, fp, fn):
    
    if vp + vn + fp + fn != 0:
        return (vp + vn) / (vp + vn + fp + fn)
    
    return 0

print(acuracia(vp, vn, fp, fn))

0.75


$$ \text{Precisão(Precision)} = \frac{VP}{VP + FP} $$

In [843]:
def precisao (vp, fp):
    
    if vp + fp != 0:
        return vp / (vp + fp)
    
    return 0
    

precision = precisao(vp, fp)
print(precision)

1.0


$$ \text{Revocação(Recall)} = \frac{VP}{VP + FN} $$

In [844]:
def recall(vp, fn):
    if vp + fn != 0:
        return vp / (vp + fn)
    
    return 0

rec = recall(vp, fn)
print(rec)

0.5


$$ \text{F1-score} = 2 . \frac{\text{Precisão . Recall}}{\text{Precisão + Recall}}$$

In [845]:
def f1_score(precision, recall):
    if precision + recall != 0:
        return 2 * ((precision * recall) / (precision + recall))

    return 0

print(f1_score(precision, rec))

0.6666666666666666
